# QFF2026 QML Track — PS-Q2 reproducible workflow

Uses seed 42. Cached ideal-kernel tables are loaded by default; set `RECOMPUTE_IDEAL_ABLATION = True` to recompute the full ideal grid. Quantum noise/Task A/Task B require the pinned Qiskit + Aer environment. This notebook records dependency blockers rather than substituting invented results.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
RECOMPUTE_IDEAL_ABLATION = False
SEED = 42
print('Root:', ROOT, '| seed:', SEED)

## 1. Data preparation

In [ ]:
from src.data import load_q2_data
X_train, y_train, X_test, sample_ids, label_to_int, int_to_label = load_q2_data(ROOT)
print('Train shape:', X_train.shape, '| Test feature shape:', X_test.shape)
print('Label mapping:', label_to_int, '| Train label counts:', y_train.value_counts().to_dict())
assert 'sample_id' not in X_test.columns and len(sample_ids) == 200

## 2. Classical baselines (cached Prompt 2 results)

In [ ]:
classical_path = ROOT / 'results/tables/classical_baselines.csv'
classical = pd.read_csv(classical_path)
display(classical[['model','balanced_accuracy_mean','balanced_accuracy_std','f1_mean','f1_std']].sort_values('balanced_accuracy_mean', ascending=False))

## 3. Ideal quantum-kernel ablation and diagnostics

In [ ]:
from src.quantum_kernel import run_ideal_kernel_ablation, print_top_configs
ideal_path = ROOT / 'results/tables/ideal_kernel_ablation.csv'
if RECOMPUTE_IDEAL_ABLATION or not ideal_path.exists():
    ideal = run_ideal_kernel_ablation(ROOT)
else:
    ideal = pd.read_csv(ideal_path)
    print('Loaded cached ideal-kernel ablation:', ideal.shape)
print_top_configs(ideal, ROOT)
display(ideal.sort_values('cv_balanced_accuracy', ascending=False).head(5))

## 4. Task A — official Bell benchmark

In [ ]:
try:
    from src.task_a import run_task_a
    task_a_result = run_task_a(ROOT)
except Exception as exc:
    print(f'Task A NOT MEASURED in this runtime: {type(exc).__name__}: {exc}')
    print('See results/task_a_run.log and results/task_a_explanation.md. No values are fabricated.')

## 5. Task B — Processor D

In [ ]:
try:
    import qiskit, qiskit_aer
except Exception as exc:
    candidate_path = ROOT / 'results/tables/processor_d_candidates.csv'
    if candidate_path.exists(): display(pd.read_csv(candidate_path))
    print(f'Processor D benchmark NOT MEASURED: {type(exc).__name__}: {exc}')
else:
    from src.processor_d import run_sweep
    candidates = run_sweep(ROOT)
    bench_run = subprocess.run([sys.executable, 'benchmark/qff_qml_benchmark.py', 'processor_d_config.json'], cwd=ROOT, capture_output=True, text=True)
    print(bench_run.stdout)
    if bench_run.returncode != 0: print(bench_run.stderr)

## 6. Noisy kernel analysis

In [ ]:
try:
    import qiskit, qiskit_aer
    from src.noisy_kernel import run_noisy_kernel
    noisy_result = run_noisy_kernel(ROOT, recompute=False)
except Exception as exc:
    print(f'Noisy kernel NOT RUN: {type(exc).__name__}: {exc}')
    print('Noisy metrics must be generated with the unmodified official noise model in the pinned environment.')

## 7. Final CV-selected model and test predictions

In [ ]:
from src.final_model import run as run_final_model
final_comparison = run_final_model(ROOT)
display(final_comparison)
print('Prediction rows:', len(pd.read_csv(ROOT / 'predictions_q2.csv')))

## 8. Prediction format validation

In [ ]:
import subprocess
check = subprocess.run([sys.executable, 'tools/validate_predictions.py', 'q2', 'predictions_q2.csv'], cwd=ROOT, capture_output=True, text=True)
print(check.stdout)
if check.returncode != 0: print(check.stderr)
assert check.returncode == 0, 'Prediction validation failed'

## 9. Evidence status

The notebook's successful format-validation cell does not imply that Task A, Task B, or noisy kernel simulations passed. Consult `results.md`, `results/task_a_run.log`, `results/processor_d_sweep_run.log`, and `results/noisy_kernel_run.log` for exact status.